# FTM Medical Attack - Colab Notebook

**Feature Tuning Mixup (FTM)** for targeted adversarial attacks on medical chest X-ray models.

Paper: *Improving Transferable Targeted Attacks with Feature Tuning Mixup* (CVPR 2025)

---

## 📋 Prerequisites

1. **GPU Runtime**: `Runtime` → `Change runtime type` → `GPU` (T4/V100)
2. **Data Ready**: 1000 images at 1024×1024 + attack CSV (already in project)
3. **Google Drive**: Mount to persist results (Colab disconnects after 12h)

In [ ]:
# Check GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 📦 Install Dependencies

In [ ]:
!pip install -q torch torchvision timm pillow numpy torchxrayvision pandas

# Verify imports
import torch, torchvision, timm, PIL, numpy, torchxrayvision, pandas
print("All packages installed ✓")

## 🔗 Mount Google Drive (for results persistence)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# SET THIS TO YOUR DRIVE FOLDER PATH
PROJECT_DIR = "/content/drive/MyDrive/ftm_health/ftm/feature-tuning-mixup"

print(f"Project: {PROJECT_DIR}")
%cd {PROJECT_DIR}
import os
print(f"Working in: {os.getcwd()}")
print("Contents:", os.listdir("."))

# Verify data exists
print(f"\nImages: {len([f for f in os.listdir('data/images') if f.endswith('.png')])} files")
import pandas as pd
df = pd.read_csv('data/images/images-224.csv')
print(f"Attack CSV: {len(df)} rows")
print(df['TrueLabel'].value_counts().sort_index())

## ⚡ Quick Test Run (4 images, 10 iterations)

Verify everything works before full run.

In [ ]:
!python main.py \
    --model_name xrv_chexnet \
    --save_dir ./exp/test_run \
    --num_images 4 \
    --max_iter 10 \
    --batch_size 2 \
    --eval

## 🚀 Full Attack Run (1000 images, 300 iterations)

**Expected time on T4:** ~3-4 hours
**Expected time on V100:** ~1.5-2 hours

⚠️ **Do not interrupt** - no resume support. Results saved to `--save_dir`.

In [ ]:
# FTM (ensemble=1) - batch_size=20 for faster overnight run
!python main.py \
    --model_name xrv_chexnet \
    --save_dir ./exp/medical_results \
    --batch_size 20 \
    --eval

## 🔬 FTM-E Run (Ensemble Size 2) - Optional

Better transferability, 2x compute.

In [ ]:
# FTM-E (ensemble=2) - run after FTM if needed
# !python main.py \
#     --model_name xrv_chexnet \
#     --save_dir ./exp/medical_results_ftme \
#     --ensemble_size 2 \
#     --eval

## 📊 View Results

In [ ]:
import os

results_dir = "./exp/medical_results/results"
if os.path.exists(results_dir):
    print("=== SUMMARY ===")
    with open(os.path.join(results_dir, "summary.txt")) as f:
        print(f.read())
    
    print("=== CSV Preview ===")
    import pandas as pd
    df = pd.read_csv(os.path.join(results_dir, "attack_results.csv"))
    print(df.head(10))
    print(f"\nTotal rows: {len(df)}")
    print(f"Models: {df['ModelName'].unique()}")
else:
    print("Results not found. Check save_dir.")

## 📥 Download Results to Local

In [ ]:
from google.colab import files

results_dir = "./exp/medical_results/results"
if os.path.exists(results_dir):
    files.download(os.path.join(results_dir, "summary.txt"))
    files.download(os.path.join(results_dir, "attack_results.csv"))
    print("Download started...")
else:
    print("Run the attack first.")

## 📝 Notes

- **Resolution**: Images loaded at 1024×1024, each model downscales to native (224 or 512)
- **Models**: 9 targets (4 XRV DenseNet224, 1 XRV ResNet512, 4 ImageNet)
- **No upscaling** - all models receive properly downscaled input
- **Results persist** in Google Drive at `exp/medical_results/`